# Resampling durch Interpolation

Zeitsignale, die unterschiedliche Zeitachsen besitzen, können durch Interpolation 
([`scipy.interpolate.interp1d()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.interp1d.html)) auf eine gemeinesame Zeitachse gebracht werden,um diese dann zu verrechnen zu können.

Beispiel:  Querbeschleuinigung aus Gierrate und Geschwindigkeit





---
History
<br>
2026-06-07 ug V1.3

---
### Hintergrund der Brechnung
Frage: Wie berechne ich die Querbeschleuinigung aus Gierrate und Geschwindigkeit?
<br>
Antwort [Perplexity](https://www.perplexity.ai/search/wie-berechnen-ich-die-querbesc-tMU1IFicTZiP1LLahSPXJg#0)

Um die Querbeschleunigung ($a_y$) eines Fahrzeugs aus der Gierrate ($\dot{\psi}$, englisch: yaw rate) und der Geschwindigkeit ($v$) zu berechnen, kannst du folgende Formel verwenden:

$$
a_y = v \cdot \dot{\psi}
$$

**Erklärung:**

- $a_y$: Querbeschleunigung (in m/s²)
- $v$: Geschwindigkeit des Fahrzeugs (in m/s)
- $\dot{\psi}$: Gierrate, also die Winkelgeschwindigkeit um die Hochachse (in rad/s)

Diese Formel gilt insbesondere bei stationärer Kreisfahrt, also wenn das Fahrzeug sich mit konstanter Geschwindigkeit auf einer Kreisbahn bewegt und der Schwimmwinkel (Driftwinkel) vernachlässigbar ist. In diesem Fall entspricht die Querbeschleunigung der Zentripetalbeschleunigung, die senkrecht zur Fahrtrichtung wirkt[^4][^7].

**Beispiel:**

Angenommen, ein Fahrzeug fährt mit 50 km/h ($v = 13{,}89\,\mathrm{m/s}$) und hat eine gemessene Gierrate von 0,2 rad/s:

$$
a_y = 13{,}89\,\mathrm{m/s} \cdot 0,2\,\mathrm{rad/s} = 2{,}78\,\mathrm{m/s}^2
$$

**Zusätzliche Hinweise:**

- Die Gierrate misst, wie schnell sich das Fahrzeug um seine Hochachse dreht.
- Die Geschwindigkeit muss in Meter pro Sekunde (m/s) und die Gierrate in Radiant pro Sekunde (rad/s) angegeben werden.
- Bei höheren Fahrdynamik-Anforderungen (z.B. bei großen Driftwinkeln oder instationärer Fahrt) kann die Querbeschleunigung auch von weiteren Faktoren wie dem Schwimmwinkel beeinflusst werden. 

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline

Wir haben zwei Signale mit unterschiedlichen Abtastzeiten:

- Signal1: Geschwindigkeit `velocity` wurde mit 100ms abgetastet und
- Signal2: Gierrate `yawrate` wurde mit 20ms abgestatet

Wir wollen aus dem Produkt der Geschwindigkeit und der Gierrate die Querbeschleunigung berechnen.

Hintergrund: Die Signale der Geschwindigkeit und der Gierrate wurden auf unterschiedlichen CAN-Botschaften übertragen.



### Testsignalerzeugung

unterschiedliche Abtastraten für Geschwindigkeit und Gierrate festlegen:

In [ ]:
dT_Geschwindigkeit = 0.1
dT_Gierrate = 0.02

Geschwindigkeits- und Gierratensignal erzeugen.

Auf die genaue Form und Zusammensetzung des Signals kommt es nicht an.

- Geschwindigkeit:
    - Zeitachse: `t_Geschwindigkeit`
    - Abtastwerte:  `Geschwindigkeit`
- Gierrate :
    - Zeitachse: `t_Gierrate`
    - Abtastwerte:  `Gierrate`
   

In [ ]:
t_start = 0   # Startzeitpunkt
t_end = 2     # Endzeitpunkt

t_offset = 0.01

# Zeitachsen
t_Geschwindigkeit = np.arange(t_start+t_offset,t_end,dT_Geschwindigkeit)
t_Gierrate = np.arange(t_start,t_end,dT_Gierrate)

# Abtastwerte
f0 = 0.1
Geschwindigkeit = np.sin(2*np.pi*f0*t_Geschwindigkeit)
Gierrate = np.sin(2*np.pi*f0*t_Gierrate)


### Darstellung der Abtastwerte

In [ ]:
fig, (ax1, ax2,) = plt.subplots(nrows=2,figsize=(16,8),sharex=True)


ax1.plot(t_Geschwindigkeit,Geschwindigkeit,marker='o',label='Geschwindigkeit')
ax1.grid(True)
ax1.set_title('Geschwindigkeit')
ax1.legend()

ax2.plot(t_Gierrate,Gierrate,marker='o',label='Gierrate')
ax2.grid(True)
ax2.set_title('Gierrate')
ax2.legend()
ax2.set_xlim(0,0.5);


Die Abtastwerte von der Geschwindigkeit und der Gierrate sind zeitlich unterschiedlich.

Durch Interpolation werden wir jetzt Geschwindigkeitswerte zuden Abtastzeitpunkten der Gierrate berechnen.

---
## Interpolation

Siehe scipy [scipy - interpolation - tutorial](https://docs.scipy.org/doc/scipy/reference/tutorial/interpolate.html)

Für die Interpolation der Geschwindigkeit verwenden die Funktion [`scipy.interpolate.interp1d()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.interp1d.html)
, die uns ein Objekt zurückliefert mit dem wir dann für beliebige Abtastzeitpunkt Werte für die Geschwindigkeit interpolieren können.

Bei Aufruf der Funktion [`interp1d()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.interp1d.html) werden folgende Werte übergeben:
- Bekannte Werte der Geschwindigkeit: Zeitachse und Werte
- `kind=` wie interpoliert werden soll
    - `previous` - es wird der letzte Wert gehalten (ZOH zero order hold)
    - `cubic` - es kontinuierlich interpoliert durch Verwendung eines Splines dritter Ordnung
- `bounds_error=False` und `fill_value=0.0` - außerhalb der bekannten Daten wird mit Null weiter extrapoliert





In [ ]:
from scipy.interpolate import interp1d

####  Zero order hold Interploation 
Bei der Zero order hold Interpolation, wird der letzte Abtastwert jeweils weitergeführt.
<br>
[Zero order hold](https://en.wikipedia.org/wiki/Zero-order_hold) steht für ein Halteglied 0.ter Ordnung und als ZOH abgekürzt.
<br> 
Bei einem Halteglied 1.ter Ordnung würde das Signal linear extrapoliert.

Die Zero order hold Interpolation ist bei diskreten oder Boolschen Signale sinnvoll, kann aber auch bei kontinuierlichen Signalen wie der Geschwindingkeit der Gierrate genutzt werden.

In [ ]:
Geschwindigkeit_interp1d_ZOH = interp1d(t_Geschwindigkeit,Geschwindigkeit, kind='previous',bounds_error=False,fill_value=0.0)

Jetzt können wird Geschwindigkeitswert für beliebige Abtastzeitpunkte bestimmen:

z.B. für t = 0.3 Sekunden

In [ ]:
Geschwindigkeit_interp1d_ZOH(0.3)

Es kann auch ein ganzes Array mit Werte interpoliert werden:

In [ ]:
Geschwindigkeit_interp1d_ZOH(np.array([0.3, 0.4, 0.5]))

Wir können auch das Array mit den Zeitpunkten der Gierrate verwenden, um die entsprechenden Geschwindigkeitswerte zu bestimmen:

In [ ]:
Geschwindigkeit_interp1d_ZOH(t_Gierrate)

Er besitzt jetzt die gleich Länge wie das Array mit den Werten der Gierrate

In [ ]:
Geschwindigkeit_interp1d_ZOH(t_Gierrate).shape, Gierrate.shape

Jetzt können wir die Querbeschleunigung berechnen:

In [ ]:
Querbeschleunigung_ZOH = Gierrate * Geschwindigkeit_interp1d_ZOH(t_Gierrate)
Querbeschleunigung_ZOH.shape, Gierrate.shape

Alternative kann die Geschwindigkeit kontinuierlich interpoliert werden
<br>
kind='cubic'

In [ ]:
Geschwindigkeit_interp1d_continuous = interp1d(t_Geschwindigkeit,Geschwindigkeit, kind='cubic',bounds_error=False,fill_value=0.0)

In [ ]:
Querbeschleunigung_continuous = Gierrate * Geschwindigkeit_interp1d_continuous(t_Gierrate)
Querbeschleunigung_continuous.shape, Gierrate.shape

#### Visualisierung

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(nrows=3,figsize=(16,8),sharex=True)


ax1.plot(t_Geschwindigkeit,Geschwindigkeit,color='black',marker='o',lw=3,label='Geschwindigkeit - original')
ax1.plot(t_Gierrate,Geschwindigkeit_interp1d_ZOH(t_Gierrate),color='blue',marker='o',label='Geschwindigkeit - ZOH')
ax1.plot(t_Gierrate,Geschwindigkeit_interp1d_continuous(t_Gierrate),color='red',marker='o',label='Geschwindigkeit - kontinuierlich')
ax1.grid(True)
ax1.set_title('Geschwindigkeit')
ax1.legend()
ax1.set_ylim(0,0.4);


ax2.plot(t_Gierrate,Gierrate,marker='o',color='black',label='Gierrate')
ax2.grid(True)
ax2.set_title('Gierrate')
ax2.legend()

ax3.plot(t_Gierrate,Querbeschleunigung_ZOH,marker='o',color='blue',label='Querbeschleunigung berechnet mit ZOH interpoliert')
ax3.plot(t_Gierrate,Querbeschleunigung_continuous,marker='o',color='red',label='Querbeschleunigung berechnet mit kontinuierlich interpoliert')
ax3.grid(True)
ax3.set_title('Querbeschleunigung')
ax3.legend()
ax3.set_ylim(0,0.1);

ax3.set_xlim(0.1,0.5);

